In [1]:
import pandas as pd
from sodapy import Socrata
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

/home/joe/anaconda3/envs/bic/lib/python3.6/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.18) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


In [2]:
## Get datasets on CIM
cim_url_query = 'data.colorado.gov'
datasets = None

with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()

In [ ]:
datasets[4]

In [4]:
# ## Split out dictionary of lists to a dictionary by 4x4
# cim_more = {}
# for dicts in datasets:
#     cim_more[dicts['resource']['id']] = dicts
    

## Get BIC and CIM datasets

In [3]:
## Read in our Inventory and the CIM inveontry that James created
bic = pd.read_excel("BIC Dataset Tracker.xlsx",header=2,sheet_name="PublishedData",engine="openpyxl")
cim = pd.read_excel("BIC Dataset Tracker.xlsx",header=0,sheet_name="cimAllData",engine="openpyxl")

In [4]:
print(bic.shape)
print(cim.shape)

(399, 63)
(543, 13)


In [5]:
for col in  cim.columns:
    print(col)

Title
ID
Parent ID
Type
Created At
Data Updated At
Metadata Updated At
Days Since Change
Page Views Last Week
Page Views Last Month
Page Views Total
Download Count
Weighted Average


In [6]:
## Get list of 4x4s and API 4x4s from our inventory list
bic_s4x4 = bic["Socrata Link"].to_list()
bic_a4x4 = bic["API 4x4"].to_list()

In [7]:
cim.columns

Index(['Title', 'ID', 'Parent ID', 'Type', 'Created At', 'Data Updated At',
       'Metadata Updated At', 'Days Since Change', 'Page Views Last Week',
       'Page Views Last Month', 'Page Views Total', 'Download Count',
       'Weighted Average'],
      dtype='object')

In [8]:
## Get list of 4x4s from CIM datasets
cim_4x4 = cim["ID"].to_list()
cim_p4x4 = cim["Parent ID"].to_list()

In [9]:
cim["Type"].value_counts()

dataset    367
map        157
href        13
filter       6
Name: Type, dtype: int64

## BIC Not In CIM

In [10]:
## How many datasets in BIC, not in CIM?
print(len(set(bic_s4x4) - set(cim_4x4)))
print(set(bic_s4x4) - set(cim_4x4))
bic_notCIM_ids = set(bic_s4x4) - set(cim_4x4)

6
{'35y-dxds', 'rifs-n6ib', 'ms6b-y4xc', '2mk-94p9', 'n9gj-cjub', '4yw9-a5y6'}


In [11]:
## 6 Datasets in BIC, not on CIM
bic.loc[bic["Socrata Link"].isin(bic_notCIM_ids)].to_csv("BIC_not_in_CIM.csv")

## CIM Not In BIC

In [12]:
## How many datasets in CIM, not in BIC?
len(set(cim_4x4) - set(bic_s4x4))
cim_notbic_ids = (set(cim_4x4) - set(bic_s4x4))
print(len(cim_notbic_ids))

151


In [13]:
cim_not_bic = cim.loc[cim["ID"].isin(cim_notbic_ids)]
cim_not_bic.shape

(151, 13)

In [14]:
##  What is the type of datasets?  Are they mostly maps?
cim_not_bic["Type"].value_counts()

dataset    105
map         36
filter       6
href         4
Name: Type, dtype: int64

In [15]:
##  Dataset that are not type as dataset

cim_not_bic.loc[cim_not_bic["Type"] == "dataset"].to_csv("CIM_not_in_BIC_datasets.csv",index=False)
print(cim_not_bic.loc[cim_not_bic["Type"] == "dataset"].shape)
cim_not_bic.loc[cim_not_bic["Type"] != "dataset"].to_csv("CIM_not_in_BIC_not_datasets.csv",index=False)
print(cim_not_bic.loc[cim_not_bic["Type"] != "dataset"].shape)

(105, 13)
(46, 13)


In [16]:
cim_not_bic_dataset= cim_not_bic.loc[cim_not_bic["Type"] == "dataset"]
cim_not_bic_dataset_ids = cim_not_bic_dataset["ID"].to_list()

In [17]:
len(cim_not_bic_dataset_ids)

105

In [18]:
cim_not_bic_dataset_wparent = cim.loc[cim['Parent ID'].isin(cim_not_bic_dataset_ids)]

In [37]:
cim_not_bic_dataset_wparent["Type"].value_counts()

map    74
Name: Type, dtype: int64

In [38]:
##  write out the children maps
cim_not_bic_dataset_wparent.to_csv("CIM_not_BIC_dataset_child_map.csv",index=False)

In [39]:
## Get the ids from cim_not_bic_dataset that have a parent 
cim_not_bic_dataset_wparent_ids = cim_not_bic_dataset_wparent["Parent ID"].to_list()
len(cim_not_bic_dataset_wparent_ids)

74

In [40]:
##  write out the parents of the maps
cim_not_bic_dataset.loc[cim_not_bic_dataset["ID"].isin(cim_not_bic_dataset_wparent_ids)].to_csv("CIM_not_BIC_dataset_parent_to_map.csv",index=False)

In [41]:
##  OK, get the 31 datasets that are NOT parents to other datasets... 
cim_not_bic_dataset_leftover_ids = set(cim_not_bic_dataset_ids) - set(cim_not_bic_dataset_wparent_ids) 
len(cim_not_bic_dataset_leftover_ids)

31

In [42]:
##  Get the datasets in CIM not in our INV that are listed as datasets... these are the ones we are concerted about
cim_not_bic_dataset_notparent = cim_not_bic_dataset.loc[cim_not_bic_dataset["ID"].isin(cim_not_bic_dataset_leftover_ids)]


In [43]:
cim_not_bic_dataset_notparent.to_csv("CIM_not_BIC_datasets_notparent.csv",index=False)

In [44]:
cim_not_bic_dataset_notparent["Title"].value_counts()

geometry                                                                              6
Development Projects in Grand Junction Colorado 2019                                  1
Boulder County Building Footprints                                                    1
Colorado State Special Event Sales Tax Return                                         1
Colorado State Agency Natural Gas Use FY15 - FY21                                     1
Enterprise Zones Test Jan 2019                                                        1
County of Boulder Prescription Drop Boxes                                             1
Colorado State Agency Fuel Use FY15 - FY21                                            1
Boulder County Marijuana Establishments                                               1
zip_codes_2013                                                                        1
City of Denver Athletic Fields                                                        1
City of Denver Golf Courses     

In [27]:
cim_not_bic_dataset_notparent.head(30)

,Title,ID,Parent ID,Type,Created At,Data Updated At,Metadata Updated At,Days Since Change,Page Views Last Week,Page Views Last Month,Page Views Total,Download Count,Weighted Average
103,City of Denver Traffic Accidents,cpwf-cznk,NaN,dataset,2023-11-08T17:39:52.000Z,2023-11-08T17:43:22.000Z,2023-11-21T19:58:45.000Z,71,7,26,61,4,15.0
105,Boundaries for Denver Public Schools,cyih-477r,NaN,dataset,2020-07-24T17:04:57.000Z,2023-10-05T20:43:09.000Z,2023-10-05T20:43:07.000Z,104,8,28,588,66,93.2
111,Mapping Prejudice: Denver (1931),3qth-7k3v,NaN,dataset,2023-05-23T18:04:01.000Z,2023-07-12T20:57:53.000Z,2023-07-12T21:10:53.000Z,189,4,20,206,44,43.4
115,"Colorado Art, Culture and Humanities Non-Profits",i35y-dxds,NaN,dataset,2023-05-15T19:11:11.000Z,2023-05-17T16:39:55.000Z,2023-05-17T16:39:54.000Z,246,3,16,166,78,51.9
119,North American Industry Classification System ...,i2mk-94p9,NaN,dataset,2023-04-11T15:32:21.000Z,2023-05-09T13:37:46.000Z,2023-05-09T13:37:42.000Z,254,3,18,225,22,35.8
120,Colorado State Agency Renewable Energy Use FY1...,6xzj-f3x2,NaN,dataset,2023-03-06T19:54:41.000Z,2023-03-06T19:57:34.000Z,2023-03-06T19:57:40.000Z,317,3,17,212,18,32.7
121,Colorado State Agency Fuel Use FY15 - FY21,qugj-me4q,NaN,dataset,2023-03-06T19:41:44.000Z,2023-03-06T19:47:50.000Z,2023-04-29T17:52:58.000Z,317,5,16,206,22,34.1
122,Colorado State Agency Natural Gas Use FY15 - FY21,wjq6-z3tj,NaN,dataset,2023-03-06T19:50:29.000Z,2023-03-06T19:53:52.000Z,2023-03-06T19:53:58.000Z,317,4,16,197,24,33.7
152,City of Denver Golf Courses,farp-k3rw,NaN,dataset,2022-06-27T18:02:20.000Z,2022-06-27T18:05:17.000Z,2022-06-27T18:05:23.000Z,569,4,18,326,43,54.6
153,City of Denver Trails and Sidewalks,hmjx-udkf,NaN,dataset,2022-06-27T17:57:21.000Z,2022-06-27T18:00:56.000Z,2022-06-27T18:01:03.000Z,570,7,21,289,50,55.2


In [ ]:
i35y-dxds
i2mk-94p9

### MOre Detailed CIM Inventory

In [35]:
## This file is a different inventory from CIM that has a little more info... downloaded on 01/23/24
cim_newinv = pd.read_csv("Asset_Inventory.csv")

In [29]:
cim_newinv.shape

(2645, 111)

In [32]:
a = "Dataset Title	Short Description	Category	Keywords	Type	License Type	Data Provider	Data Provided by	Source Link	State Steward	Citation	Agency Program Page	Agency Data Series Page	Business Contact and Phone	Technical Contact and Phone	Data Source	Unit of Analysis	Granularity Coverage	Geographic Extent and Division	Collection Mode	Collection Methodology	Data Collection Instrument	Date of Initial Dataset Creation	Field Names, comma delimited	Oldest Record in Dataset	Newest Record in Dataset	Long Description	Data Dictionary	Additional Metadata	Technical Documentation	Data Quality Certification	Applicable Information Quality Guideline Designation	Stewardship Plan	Collection Method	Horizontal Accuracy	Horizontal Coordinate System	Update Schedule	Update Method	Source Update Schedule	Update Type	Total Records at Initial Publish	Row Class RDF	Subject Column RDF	Single Row	Row Count (11/20/17)	Total Fields at Initial Publish	FIle Size at Initial Publish or as of 3-1-2016	Expected approximate increase in record count at update	Date Published to CIM	GoCode FY Published to CIM	Socrata Link	API 4x4	Web Display Coordinate System	Coordinate System Disclaimer	Related Datasets	Quarter of Gov FY Published	CIM Updated	Days Since CIM Update	cimAllData Updates"

In [35]:
for col in sorted(a.split("\t")):
    print(col)

API 4x4
Additional Metadata
Agency Data Series Page
Agency Program Page
Applicable Information Quality Guideline Designation
Business Contact and Phone
CIM Updated
Category
Citation
Collection Method
Collection Methodology
Collection Mode
Coordinate System Disclaimer
Data Collection Instrument
Data Dictionary
Data Provided by
Data Provider
Data Quality Certification
Data Source
Dataset Title
Date Published to CIM
Date of Initial Dataset Creation
Days Since CIM Update
Expected approximate increase in record count at update
FIle Size at Initial Publish or as of 3-1-2016
Field Names, comma delimited
Geographic Extent and Division
GoCode FY Published to CIM
Granularity Coverage
Horizontal Accuracy
Horizontal Coordinate System
Keywords
License Type
Long Description
Newest Record in Dataset
Oldest Record in Dataset
Quarter of Gov FY Published
Related Datasets
Row Class RDF
Row Count (11/20/17)
Short Description
Single Row
Socrata Link
Source Link
Source Update Schedule
State Steward
Stewards

In [31]:
for col in cim_newinv.columns:
    print(col)

UID
Name
Description
Owner
Owner UID
Publication Stage
Audience
Approval Status
Provenance
Type
Category
Tags
url
API Endpoint
Creation Date (UTC)
Last Metadata Updated Date (UTC)
Last Data Updated Date (UTC)
Domain
Derived View
Visits
Downloads
Row Label
Row Count
Column Count
Contact Email
License
Attribution
Attribution Link
Published Version Name
Published Version UID
Parent UID
Dataset Summary: Agency
Required Field Set: Primary Content Provider
Data Description: Newest Record in Dataset
Dataset Coverage: Geographic Coverage
Dataset Coverage: Unit of Analysis
Contributing Agency Information: Citation
Additional Dataset Documentation: FGDC Compliance (Geospatial Only)
Data.colorado.gov Dataset Descriptions: Time Period
Data Quality: Data Governance Plan
Geospatial: Coordinate System Disclaimer
Dataset Summary: Frequency
Geospatial: Web Display Coordinate System
Data.colorado.gov Dataset Descriptions: Stewardship Plan
Data Description: Related Datasets
Data Description: Data Collect

In [33]:
# Of the 105 datasets, how many are the Parent of datasets in New CIM inventory
cim_notinv_ds_Pid = cim_newinv.loc[cim_newinv["Parent UID"].isin(cids)]

In [34]:
cim_notinv_ds_Pid.shape

(80, 111)

In [36]:
##   Of teh 80 that are Children (i.e. Their parent id is one of the 105 not in our inventory), 
##   what kind of datasets are they
cim_notinv_ds_Pid['Type'].value_counts()

map    80
Name: Type, dtype: int64

In [39]:
##  Which of the 105 datases are NOT the Parents to another dataset
a = set(cids) - set(cim_notinv_ds_Pid["Parent UID"].to_list())

In [40]:
##  Roh-Roh, there are too many.  There should be just 25 (105-80)
len(a)

29

In [41]:
##  Ah, OK, some are parents to multiple datasets
cim_notinv_ds_Pid["Parent UID"].value_counts()

49x6-nvb5    3
dwez-qzwp    2
q9ua-9k3d    2
8d8p-eabg    1
grnx-2zwm    1
jppe-46v2    1
rz4c-na6y    1
93kh-r488    1
m3j7-raj9    1
n5xr-summ    1
umwy-67rm    1
5733-dk6i    1
i3mi-mx9y    1
k73m-yqwy    1
bv6m-vkmf    1
k9xv-8qd6    1
tsdg-z9uy    1
2czt-qzar    1
vy7p-ixjg    1
ivcq-5wnv    1
8hh9-tn7j    1
4mk5-vv45    1
kd4i-r65w    1
kyzd-u4zr    1
3yjp-6a33    1
jfqj-df4i    1
u47d-dmww    1
7e99-m9fd    1
etbc-tkgn    1
f4n4-vnyx    1
vefd-xfvu    1
wshk-29g7    1
2f98-hukf    1
uzap-vkkz    1
7jyi-58ux    1
vx8e-g83x    1
85u5-cuz2    1
gx4f-gkav    1
t48m-528x    1
82e7-gztd    1
385d-6w27    1
vdki-aa5n    1
ws3z-urxq    1
imy5-eits    1
rue4-ebee    1
rxks-zb9z    1
2rrq-heiy    1
bvd7-vs7t    1
wwwp-4j5y    1
ehbb-wpcw    1
594w-hhep    1
a5gi-9pma    1
ibr4-saca    1
xf9x-4vrx    1
3jv8-fthf    1
7qc8-fj3p    1
cpwf-cznk    1
rt3s-2378    1
sbcg-h3as    1
mwv9-ygsf    1
jiwb-nqtz    1
q2zb-n3df    1
gtau-pbhu    1
adbb-iu37    1
a85s-zxvh    1
pgs5-k2ve    1
kt3p-393c 

In [45]:
## Of the 80 datasets, how many are maps?
cim_newinv.loc[cim_newinv["UID"].isin(a)]

,UID,Name,Description,Owner,Owner UID,Publication Stage,Audience,Approval Status,Provenance,Type,Category,Tags,url,API Endpoint,Creation Date (UTC),Last Metadata Updated Date (UTC),Last Data Updated Date (UTC),Domain,Derived View,Visits,Downloads,Row Label,Row Count,Column Count,Contact Email,License,Attribution,Attribution Link,Published Version Name,Published Version UID,Parent UID,Dataset Summary: Agency,Required Field Set: Primary Content Provider,Data Description: Newest Record in Dataset,Dataset Coverage: Geographic Coverage,Dataset Coverage: Unit of Analysis,Contributing Agency Information: Citation,Additional Dataset Documentation: FGDC Compliance (Geospatial Only),Data.colorado.gov Dataset Descriptions: Time Period,Data Quality: Data Governance Plan,Geospatial: Coordinate System Disclaimer,Dataset Summary: Frequency,Geospatial: Web Display Coordinate System,Data.colorado.gov Dataset Descriptions: Stewardship Plan,Data Description: Related Datasets,Data Description: Data Collection Instrument,Data Quality: Applicable Information Quality Guideline Designation,Dataset Summary: Source,Dataset Summary: Suggested by Public,Required Field Set: Secondary Content Editor,Data Description: Collection Method,Data.colorado.gov Dataset Descriptions: Sub agency,Data Updates: Total Records at Recent Update,Geo Data Information: Positional Accuracy,Dataset Summary: Local Government Entity,Data.colorado.gov Dataset Descriptions: Frequency of Update,Data.colorado.gov Dataset Descriptions: Data Quality Certified,Geospatial: Positional Accuracy,Contributing Agency Information: Business Contact and Phone,Data.colorado.gov Dataset Descriptions: Purpose,Contributing Agency Information: Technical Contact and Phone,Data.colorado.gov Dataset Descriptions: Secondary Content Provider,Dataset Summary: Currentness Reference,Geo Data Information: Horizontal Coordinate System,Data Quality: Expected Update Frequency,Data Description: Long Description,Data.colorado.gov Dataset Descriptions: Agency,Contributing Agency Information: Agency Data Series Page,Data Updates: Update Method,Data Updates: Update Type,Required Field Set: Agency,Dataset Summary: High Value Dataset,Geospatial: Horizontal Accuracy,Additional Dataset Documentation: Technical Documentation,Geospatial: Horizontal Coordinate System,Data Updates: Date of Inital Publish,Dataset Summary: Time Period,Data.colorado.gov Dataset Descriptions: Date Created,"Data Description: Field Names, comma delimited",Dataset Coverage: Granularity,Dataset Summary: Date Released,Additional Dataset Documentation: Additional Metadata,Contributing Agency Information: Agency Program Page,Data Updates: Total Records at Initial Publish,Contributing Agency Information: Data Source,Data.colorado.gov Dataset Descriptions: Last Updated,Data Quality: Privacy and Confidentiality,Dataset Summary: Purpose,Data Updates: File Size at Recent Update,Dataset Summary: Type,Dataset Summary: Date Updated,Data.colorado.gov Dataset Descriptions: Data Governance Plan,Data Quality: Data Quality Certification,Data Updates: Source Update Schedule,Flood 2013: Content type,Geo Data Information: Bounding Coordinates,Data Description: Single Row,Data.colorado.gov Dataset Descriptions: Status,Required Field Set: Frequency of Update,Geospatial: Collection Method,Dataset Summary: Category,Data Quality: Stewardship Plan,Data Updates: Update Schedule,Data Updates: Total Columns at Recent Update,Dataset Summary: Progress,Additional Dataset Documentation: Data Dictionary,Department Metrics: Publishing Department,Data Description: Oldest Record in Dataset,Dataset Summary: Sub Agency,Data Description: Date of Initial Dataset Creation,Data Description: Collection Mode
249,3qth-7k3v,Mapping Prejudice: Denver (1931),Real estate documents issued by the City and C...,Colorado Information Marketplace,8cet-tw9x,published,public,approved,official,dataset,Housing,"bic,covenants,deeds,gis,gocode,mapping prejudi...",https://data.colorado.gov/d/3qth-7k3v,https:/

In [52]:
cim_inv_notinvP_ids = cim_inv_notinvP['UID'].to_list()

In [66]:
# of the 80 datasets, how  many of THESE ARE in our inventory?
inv.loc[inv["Socrata Link"].isin(cim_inv_notinvP_ids)].shape


(72, 63)

### Datasets Not in CIM That ARE NOT a Parent to Another Dataset

Of the 105 datasets in CIM not in our Inventory, 25 are NOT a parent to another Dataset... which ones?

In [72]:
cim_inv_notinvP.shape

(80, 111)

In [74]:
cim_inv_notinvP_IDS = cim_inv_notinvP['UID'].to_list()

In [77]:
len(cim_inv_notinvP_IDS)

80

In [75]:
cids_notfound=[]
for id in cids:
    if id not in cim_inv_notinvP_IDS:
        cids_notfound.append(id)

In [76]:
len(cids_notfound)

105

In [39]:
!pip install oauth2client

  Using cached oauth2client-4.1.3-py2.py3-none-any.whl (98 kB)
  Using cached httplib2-0.22.0-py3-none-any.whl (96 kB)


In [2]:
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials

In [3]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
   

    df = pd.DataFrame(tracker.get_all_records(head=3))

    return df
            
df = getXrefs()

In [4]:
for col in sorted(df.columns):
    print(col)

API 4x4
Additional Metadata
Agency Data Series Page
Agency Program Page
Applicable Information Quality Guideline Designation
Business Contact and Phone
CIM Updated
Category
Citation
Collection Method
Collection Methodology
Collection Mode
Complexity
Coordinate System Disclaimer
Data Collection Instrument
Data Dictionary
Data Provided by
Data Provider
Data Quality Certification
Data Source
Dataset Title
Date Published to CIM
Date of Initial Dataset Creation
Days Since CIM Update
Expected approximate increase in record count at update
FIle Size at Initial Publish or as of 3-1-2016
Field Names, comma delimited
Geographic Extent and Division
GoCode FY Published to CIM
Granularity Coverage
Horizontal Accuracy
Horizontal Coordinate System
Keywords
License Type
Long Description
Newest Record in Dataset
Oldest Record in Dataset
Quarter of Gov FY Published
Related Datasets
Row Class RDF
Row Count (11/20/17)
Short Description
Single Row
Socrata Link
Source Link
Source Update Schedule
State Stewa

In [8]:
len(df["Socrata Link"].to_list())

399

In [ ]:
for w4x4 in df["Socrata Link"]:
    single_row = df.loc[df["Socrata Link"] == w4x4]
    tag_list = str(single_row["Keywords"].item())
    tags = tag_list.split(",")
    print(w4x4)
    data = {"name": single_row["Dataset Title"].item(),
                "category": single_row["Category"].item(),
                "attribution": single_row["State Steward"].item(),
                "license": str(single_row["License Type"].item()),
                "attributionLink": single_row["Agency Program Page"].item(),
                "description": single_row["Short Description"].item(),
                "tags": tags,
                "customFields": {
                    "Data Updates": {
                        "Update Schedule": single_row["Update Schedule"].item(),
                        "Update Method": single_row["Update Method"].item(),
                        "Update Type": single_row["Update Type"].item(),
                        "Source Update Schedule": single_row["Source Update Schedule"].item(),
                        "Total Records At Initial Publish": single_row["Total Records at Initial Publish"].item()
                    },
                    "Dataset Coverage": {
                        "Unit of Analysis": single_row["Unit of Analysis"].item(),
                        "Granularity": single_row["Granularity Coverage"].item(),
                        "Geographic Coverage": single_row["Geographic Extent and Division"].item()
                        },
                    "Geospatial": {
                        "Collection Method": single_row["Collection Method"].item(),
                        "Horizontal Accuracy": single_row["Horizontal Accuracy"].item(),
                        "Horizontal Coordinate System": single_row["Horizontal Coordinate System"].item(),
                        "Web Display Coordinate System": single_row["Web Display Coordinate System"].item(),
                        "Coordinate System Disclaimer": single_row["Coordinate System Disclaimer"].item()
                    },  
                    "Additional Dataset Documentation": {
                        "Data Dictionary": single_row["Data Dictionary"].item(),
                        "Additional Metadata": single_row["Additional Metadata"].item(),
                        "Technical Documentation": single_row["Technical Documentation"].item()
                    },
                    "Data Description": {
                        "Single Row": single_row["Single Row"].item(),
                        "Long Description": single_row["Long Description"].item(),
                        "Collection Mode": single_row["Collection Mode"].item(),
                    #    "Collection Method": single_row["Collection Methodology"].item(),
                        "Data Collection Instrument": single_row["Data Collection Instrument"].item(),
                        "Date of Initial Dataset Creation": single_row["Date of Initial Dataset Creation"].item(),
                        "Field Names, comma delimited": single_row["Field Names, comma delimited"].item(),
                        "Oldest Record in Dataset": single_row["Oldest Record in Dataset"].item(),
                        "Newest Record in Dataset": single_row["Newest Record in Dataset"].item()
                    },
                    "Data Quality": {
                        "Expected Update Frequency": single_row["Update Type"].item()
                    },
                    "Contributing Agency Information": {
                        "Citation": single_row["Citation"].item(),
                        "Agency Program Page": single_row["Agency Program Page"].item(),
                        "Agency Data Series Page": single_row["Agency Data Series Page"].item(),
                        "Data Source": single_row["Data Source"].item()
                    },
                }}
    putHist(data)

In [28]:
hist={}
def putHist(data):
    for k,v in data.items():
       # print(k,type(v))
        if isinstance(v,list):
            # for vals in v:
            #     print("   ",type(vals),vals)
            if k not in hist:
                hist[k] = []
            hist[k].append(v)
        elif isinstance(v,dict):
            for k2,v2 in v.items():
                # print("   ",k2,type(v2),v2)
                if isinstance(v2,dict):
                   for k3,v3 in v2.items():
                       # print("      ",k3,type(v3),v3)
                       if k3 not in hist:
                          hist[k3] = []
                       hist[k3].append(v3)
                else:
                    if k2 not in hist:
                        hist[k2] = []
                    hist[k2].append(v2)
        else:
            if k not in hist:
                hist[k] = []
            hist[k].append(v)

In [31]:
for key,val in hist.items():
    print(key,len(val))

name 399
category 399
attribution 399
license 399
attributionLink 399
description 399
tags 399
Update Schedule 399
Update Method 399
Update Type 399
Source Update Schedule 399
Total Records At Initial Publish 399
Unit of Analysis 399
Granularity 399
Geographic Coverage 399
Collection Method 399
Horizontal Accuracy 399
Horizontal Coordinate System 399
Web Display Coordinate System 399
Coordinate System Disclaimer 399
Data Dictionary 399
Additional Metadata 399
Technical Documentation 399
Single Row 399
Long Description 399
Collection Mode 399
Data Collection Instrument 399
Date of Initial Dataset Creation 399
Field Names, comma delimited 399
Oldest Record in Dataset 399
Newest Record in Dataset 399
Expected Update Frequency 399
Citation 399
Agency Program Page 399
Agency Data Series Page 399
Data Source 399


In [33]:
tmp = pd.DataFrame(hist)

In [35]:
for col in tmp.columns:
    if col != "tags":
        print(col,tmp[col].nunique())

name 399
category 24
attribution 37
license 3
attributionLink 75
description 294
Update Schedule 9
Update Method 6
Update Type 10
Source Update Schedule 8
Total Records At Initial Publish 183
Unit of Analysis 11
Granularity 3
Geographic Coverage 14
Collection Method 7
Horizontal Accuracy 5
Horizontal Coordinate System 23
Web Display Coordinate System 4
Coordinate System Disclaimer 2
Data Dictionary 10
Additional Metadata 31
Technical Documentation 4
Single Row 236
Long Description 245
Collection Mode 38
Data Collection Instrument 29
Date of Initial Dataset Creation 58
Field Names, comma delimited 220
Oldest Record in Dataset 92
Newest Record in Dataset 68
Expected Update Frequency 10
Citation 37
Agency Program Page 75
Agency Data Series Page 110
Data Source 25


In [38]:
#tmp["Newest Record in Dataset"].value_counts()
#tmp["Newest Record in Dataset"].value_counts()
tmp["Granularity"].value_counts()

                                     397
Parcel Level                           1
Record level transaction activity      1
Name: Granularity, dtype: int64

In [36]:
cim_newinv.columns

Index(['UID', 'Name', 'Description', 'Owner', 'Owner UID', 'Publication Stage',
       'Audience', 'Approval Status', 'Provenance', 'Type',
       ...
       'Data Quality: Stewardship Plan', 'Data Updates: Update Schedule',
       'Data Updates: Total Columns at Recent Update',
       'Dataset Summary: Progress',
       'Additional Dataset Documentation: Data Dictionary',
       'Department Metrics: Publishing Department',
       'Data Description: Oldest Record in Dataset',
       'Dataset Summary: Sub Agency',
       'Data Description: Date of Initial Dataset Creation',
       'Data Description: Collection Mode'],
      dtype='object', length=111)

In [ ]:
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
#SERVICE_ACCOUNT_FILE = os.path.join(bic_etl_home, 'general', 'metadata_updater', 'scripts', "creds.json")
SERVICE_ACCOUNT_FILE = os.path.join("home","joe","work""creds.json")

creds = None
creds = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE, scopes=SCOPES)
#sheet_id = '137W1gqkXdgTJ0mDn7Tkj2Q5DMHDPru3A_1E0pTpm9nE'
sheet_id = '1B2PvO4Q3eegdgPKG297SUA1i_b1nj7Ltq2SBoJEDGmY'

sheet_range = 'Sheet1!A4:BH402'
service = build('sheets', 'v4', credentials=creds)

# Call the Sheets API
sheet = service.spreadsheets()
result = sheet.values().get(spreadsheetId=sheet_id, range=sheet_range).execute()
values = result.get('values', [])
metadata_df = pd.DataFrame(values) #create df from the list
new_header = metadata_df.iloc[0]
metadata_df = metadata_df[1:]
metadata_df.columns = new_header

In [8]:
datasets[4]['resource']

{'name': 'DWR Well Application Permit',
 'id': 'wumm-7awb',
 'parent_fxf': [],
 'description': 'All well applications and permits issued.',
 'attribution': None,
 'attribution_link': None,
 'contact_email': None,
 'type': 'dataset',
 'updatedAt': '2024-05-30T12:10:07.000Z',
 'createdAt': '2013-11-15T18:03:41.000Z',
 'metadata_updated_at': '2018-02-14T23:12:26.000Z',
 'data_updated_at': '2024-05-30T12:10:07.000Z',
 'page_views': {'page_views_last_week': 21,
  'page_views_last_month': 438,
  'page_views_total': 71148,
  'page_views_last_week_log': 4.459431618637297,
  'page_views_last_month_log': 8.778077129535358,
  'page_views_total_log': 16.118555859348582},
 'columns_name': ['IDKey',
  'Modified',
  'Associated Case Numbers',
  'Top Perforated Casing',
  'Well Depth',
  'Elevation',
  'Associated Uses',
  'Pump Installed',
  'Permit Expires',
  'Permit Issued',
  'Address',
  'Parcel Name',
  'Location Accuracy',
  'Longitude',
  'UTM y',
  'CoordsNS',
  'Section',
  'Range',
  'Town

In [10]:
cols = datasets[4]['resource']['columns_name']
dtype = datasets[4]['resource']['columns_datatype']

In [12]:
hist = {}
for nn,col in enumerate(cols):
#    print(nn,col,dtype[nn])
    dt = dtype[nn]
    if dt not in hist:
        hist[dt]=0
    hist[dt]+=1

In [13]:
hist

{'Text': 34, 'Calendar date': 8, 'Number': 15, 'Point': 1, 'URL': 1}

In [15]:
datasets[4]['resource']['name']

'DWR Well Application Permit'

In [26]:
hist={}
dtypes={}
for dataset in datasets:
    if dataset['resource']['name'].lower()[:6] == 'census':
        print(dataset['resource']['name'])
        name=dataset['resource']['name']
        hist[name]={}
        cols = dataset['resource']['columns_name']
        dtype = dataset['resource']['columns_datatype']
        for nn,col in enumerate(cols):
            dt = dtype[nn]
            if dt in hist[name]:
                hist[name][dt]+=1
            else:
                hist[name][dt]=1
            if dt in dtypes:
                dtypes[dt]+=1
            else:
                dtypes[dt]=1
        

Census Counties in Colorado 2010
Census Blocks in Colorado 2010
Census Datasets on Colorado Information Marketplace
Census Zip Codes in Colorado 2016
Census Tracts in Colorado 2014
Census Zip Codes in Colorado 2010
Census State in Colorado 2012
Census Tracts in Colorado 2016
Census Tracts in Colorado 2010
Census in Colorado 2014
Census State in Colorado 2011
Census Zip Codes in Colorado 2018
Census Block Groups in Colorado 2010
Census Tracts in Colorado 2011
Census School Districts in Colorado 2010
Census Block Groups SF1 in Colorado 2000
Census Places SF3 in Colorado 2000
Census State in Colorado 2010
Census Zip Codes in Colorado 2014
Census Tracts SF1 in Colorado 2000
Census Tracts SF3 in Colorado 2000
Census Zip Codes SF1 in Colorado 2000
Census API in United States 1980, 1990, 2000, 2010, and ACS
Census Blocks in Colorado 2000
Census State of Colorado 2013
Census Counties SF1 in Colorado 2000
Census Core Based Statistical Area in Colorado 2011
Census State SF3 in Colorado 2000
Cens

In [27]:
names=[]
vals={}
for dt in dtypes:
    vals[dt]=[]
for name in hist:
    for dt in dtypes:
        if dt in hist[name]:
            cnt = hist[name][dt]
        else:
            cnt=0
        vals[dt].append(cnt)
    names.append(name)

In [28]:
len(names)

121

In [29]:
for dt in vals:
    print(len(vals[dt]))

121
121
121
121


In [30]:
dtypes

{'Text': 2656, 'URL': 1, 'MultiPolygon': 112, 'Number': 15760}

In [36]:
for name in sorted(hist.keys()):
    totl = 0
    for dt in hist[name]:
        totl+=hist[name][dt]
    if 'Text' in hist[name] and  hist[name]['Text']/totl > .2:
        print(name,hist[name])

Census Block Groups in Colorado 2011 {'Text': 84, 'Number': 182, 'MultiPolygon': 1}
Census Block Groups in Colorado 2012 {'Number': 107, 'Text': 49, 'MultiPolygon': 1}
Census Block Groups in Colorado 2013 {'Number': 107, 'Text': 49, 'MultiPolygon': 1}
Census Block Groups in Colorado 2014 {'Number': 107, 'Text': 49, 'MultiPolygon': 1}
Census Block Groups in Colorado 2015 {'Number': 106, 'Text': 50, 'MultiPolygon': 1}
Census Block Groups in Colorado 2016 {'Number': 106, 'Text': 50, 'MultiPolygon': 1}
Census Block Groups in Colorado 2017 {'Number': 178, 'Text': 79, 'MultiPolygon': 1}
Census Block Groups in Colorado 2018 {'Number': 177, 'Text': 79, 'MultiPolygon': 1}
Census Block Groups in Colorado 2019 {'Number': 177, 'Text': 79, 'MultiPolygon': 1}
Census Congressional Districts in Colorado 2016 {'Text': 155, 'MultiPolygon': 1}
Census Congressional Districts in Colorado 2018 {'Text': 155, 'MultiPolygon': 1}
Census Congressional Districts in Colorado 2019 {'Text': 155, 'MultiPolygon': 1}
C

In [ ]:
for dataset in datasets:
    if dataset['resource']['name'] == "Census Block Groups in Colorado 2017":
        display(dataset)
        print("----------------------------------------------")
        print("----------------------------------------------")        
        print("----------------------------------------------")        
        print("----------------------------------------------")        

In [39]:
hist['Census Block Groups in Colorado 2017']

{'Number': 178, 'Text': 79, 'MultiPolygon': 1}

In [43]:
titles={}
for dataset in datasets:
    name = dataset['resource']['name']
    if name in titles:
        print(name)
        titles[name]+=1
    else:
        titles[name]=1

Transparency Online Project (TOPS) - State Government Revenue and Expenditures in Colorado
FirstNet Governing Body
Poudre River
Map of CDHS Locations
Div 1, District 2
Combined Roads
Rio
Rio
Colorado Telehealth Network - CTN 1 Connection Locations
GrandCountyWells
Delta County New Businesses
Water Right Net Amounts
Comminity Anchor Institutions in Colorado 2016
Colorado Information Marketplace
test2
Englewood CO Businesses
Well Information Region 3
chapman well search
Delta County New Businesses
ByLocation - Based on Business Entities in Colorado
Custer_wells constructed
Untitled Visualization - Based on Business Entities in Colorado
Untitled Visualization - Based on Colorado Licensed Child Care Facilities Report
8/13-8/31 New Western Colorado Businesses
Untitled Visualization - Based on Business Entities in Colorado
Untitled Visualization - Based on Colorado Licensed Child Care Facilities Report
Untitled Visualization - Based on Business Entities in Colorado
Untitled Visualization - B